## 1 ) Importar bibliotecas

In [1]:
import polars as pl
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from pprint import pprint

## 2) Ler base de dados

In [2]:
dataset = pl.read_parquet(
    source = "../../databases/cleanned_data/diabetes_dataset.parquet"
)

print(dataset.shape)
dataset.head(2)

(442, 11)


age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
i8,i8,f32,f32,f32,f32,f32,f32,f32,f32,f32
59,2,32.099998,101.0,157.0,93.199997,38.0,4.0,4.8598,87.0,151.0
48,1,21.6,87.0,183.0,103.199997,70.0,3.0,3.8918,69.0,75.0


## 3) Preparando <code>GridSearch</code>

### 3.1) Pipeline

#### 3.1.1) Pré-processamentos

In [3]:
column_transformer1 = ColumnTransformer(
    transformers = [
        ("standardScaler", StandardScaler(), ["age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]),
        ("oneHotEncoder", OneHotEncoder(drop = "if_binary"), ["sex"])
    ]
)

column_transformer2 = ColumnTransformer(
    transformers = [
        ("standardScaler", PowerTransformer("box-cox"), ["age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]),
        ("oneHotEncoder", OneHotEncoder(drop = "if_binary"), ["sex"])
    ]
)

column_transformer3 = ColumnTransformer(
    transformers = [
        ("standardScaler", PowerTransformer("yeo-johnson"), ["age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]),
        ("oneHotEncoder", OneHotEncoder(drop = "if_binary"), ["sex"])
    ]
)

#### 3.1.2) Modelos

In [4]:
models = [
    Ridge(), Lasso(), ElasticNet()
]

#### 3.1.3) Criando combinações de Pipeline

In [5]:
pipeline = Pipeline(
    steps = [
        ("transformer", column_transformer1),
        ("model", models[0])
    ]
)

### 3.2) TargetTransformer

In [6]:
target_transformer1 = TransformedTargetRegressor(
    regressor = pipeline,
    transformer = PowerTransformer("box-cox")
)

In [7]:
# Vamos fazer um print aqui só para entender quais são os parâmetros a serem trabalhados pelo GridSearchCV:
pprint(
    target_transformer1.get_params()
)

{'check_inverse': True,
 'func': None,
 'inverse_func': None,
 'regressor': Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('standardScaler',
                                                  StandardScaler(),
                                                  ['age', 'bmi', 'bp', 's1',
                                                   's2', 's3', 's4', 's5',
                                                   's6']),
                                                 ('oneHotEncoder',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['sex'])])),
                ('model', Ridge())]),
 'regressor__memory': None,
 'regressor__model': Ridge(),
 'regressor__model__alpha': 1.0,
 'regressor__model__copy_X': True,
 'regressor__model__fit_intercept': True,
 'regressor__model__max_iter': None,
 'regressor__model__positive': False,
 'regressor__model__random_state': None,
 'regresso

In [ ]:
grid_search = GridSearchCV(
    estimator = target_transformer1,
    param_grid = {
        "regressor__transformer": [column_transformer1, column_transformer2, column_transformer3],
        "transformer": [PowerTransformer("box-cox"), PowerTransformer("yeo-johnson"), StandardScaler(), None],
        "regressor__model__alpha": [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.8, 0.9, 1, 1.1, 1.2, 1.3, 1.5, 2, 2.5, 5],
        "regressor__model": models,
    },
    cv = 5,
    n_jobs = -1,
    verbose = 1,
    scoring = [
        'neg_mean_squared_error', 'r2'
    ],
    refit = 'r2',
    error_score = "raise",
)

In [15]:
grid_search

,estimator,TransformedTa...od='box-cox'))
,param_grid,"{'regressor__model': [Ridge(), Lasso(), ...], 'regressor__model__alpha': [0.01, 0.05, ...], 'regressor__transformer': [ColumnTransfo...'), ['sex'])]), ColumnTransfo...'), ['sex'])]), ...], 'transformer': [PowerTransfor...hod='box-cox'), PowerTransformer(), ...]}"
,scoring,"['neg_mean_squared_error', 'r2']"
,n_jobs,-1
,refit,'r2'
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,False
,transformers,"[('standardScaler', ...), ('oneHotEncoder', ...)]"


### 3.3) Treinando <code>GridSearchCV</code>:

In [16]:
X = dataset.drop("target")
y = dataset["target"]

print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [17]:
grid_search.fit(
    X = X,
    y = y,
)

Fitting 5 folds for each of 648 candidates, totalling 3240 fits


,estimator,TransformedTa...od='box-cox'))
,param_grid,"{'regressor__model': [Ridge(), Lasso(), ...], 'regressor__model__alpha': [0.01, 0.05, ...], 'regressor__transformer': [ColumnTransfo...'), ['sex'])]), ColumnTransfo...'), ['sex'])]), ...], 'transformer': [PowerTransfor...hod='box-cox'), PowerTransformer(), ...]}"
,scoring,"['neg_mean_squared_error', 'r2']"
,n_jobs,-1
,refit,'r2'
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,'raise'
,return_train_score,False
,transformers,"[('standardScaler', ...), ('oneHotEncoder', ...)]"


In [18]:
grid_search.best_params_

{'regressor__model': Lasso(),
 'regressor__model__alpha': 0.05,
 'regressor__transformer': ColumnTransformer(transformers=[('standardScaler', StandardScaler(),
                                  ['age', 'bmi', 'bp', 's1', 's2', 's3', 's4',
                                   's5', 's6']),
                                 ('oneHotEncoder',
                                  OneHotEncoder(drop='if_binary'), ['sex'])]),
 'transformer': None}

In [19]:
grid_search.best_score_

np.float64(0.48246058802589165)

In [20]:
grid_search.cv_results_

{'mean_fit_time': array([0.04534936, 0.02315664, 0.01461568, 0.01011596, 0.21622663,
        0.18698406, 0.17750554, 0.18381786, 0.07314696, 0.05016212,
        0.03721676, 0.0425734 , 0.04282746, 0.02129626, 0.01126585,
        0.01246762, 0.22376456, 0.18841767, 0.19195533, 0.19338841,
        0.07443905, 0.04948964, 0.04018307, 0.03987021, 0.04450779,
        0.01917663, 0.00990319, 0.01085114, 0.21825228, 0.20333538,
        0.18406348, 0.19801126, 0.07106442, 0.04323406, 0.04247284,
        0.04149566, 0.06080546, 0.03526049, 0.01862836, 0.01756659,
        0.2442564 , 0.2365747 , 0.19371691, 0.19624071, 0.08272009,
        0.05059457, 0.04392357, 0.04142299, 0.04884973, 0.01737556,
        0.01201434, 0.01779528, 0.28235698, 0.2244565 , 0.23322921,
        0.23677783, 0.0929163 , 0.05965295, 0.05370607, 0.04597373,
        0.05134878, 0.03516498, 0.05777936, 0.03111172, 0.31573596,
        0.25777411, 0.2706718 , 0.23694263, 0.09195886, 0.06390204,
        0.06662769, 0.06974845,